# Walk-Forward Results — What the Columns Mean

Each row = **one walk-forward window** for one `max_num_components`.

### 1. Periods

- `calibration_start`, `calibration_end`  
  Historical data used for **both Stage-1 and Stage-2 optimization**.

- `test_start`, `test_end`  
  The following **out-of-sample (OOS)** period. It is not used for threshold selection or optimization.

---

### 2. Stage 1 — Broad Monte Carlo

Stage 1 performs `N_SIMULATIONS_STAGE1` random threshold combinations over the **complete threshold space**.

For every sampled threshold pair, the strategy is evaluated on the calibration period.

- `stage1_robust_buy_thr`
- `stage1_robust_sell_thr`

  → Robust center thresholds derived from the **top-performing 10% of Stage-1 simulations**.

- `stage1_top10_buy_std`
- `stage1_top10_sell_std`

  → Standard deviation of the buy/sell thresholds within that Stage-1 top-10% region.

Together, these describe the **promising threshold region identified by Stage 1**.

---

### 3. Stage 2 — Focused Monte Carlo

Stage 2 does **not** search the complete threshold space again.

It uses the Stage-1 robust thresholds as the **center of a local search region**, with the Stage-1 top-10% dispersion helping determine the width of that region.

`N_SIMULATIONS_STAGE2` new threshold combinations are sampled and evaluated on the same calibration period.

- `stage2_buy_min`, `stage2_buy_max`
- `stage2_sell_min`, `stage2_sell_max`

  → The threshold boundaries used for the Stage-2 focused search.

- `stage2_n`

  → Number of valid Stage-2 simulations performed.

Stage 2 therefore answers:

> **Which thresholds work best inside the promising region identified by Stage 1?**

---

### 4. Stage 2 — Four Selection Criteria

The **same Stage-2 simulations** are evaluated using four different selection criteria.

Each criterion selects its own threshold pair from the Stage-2 simulations.

#### Sharpe

Selects the Stage-2 threshold combination with the highest **Sharpe ratio**.

- `sharpe_buy_thr`
- `sharpe_sell_thr`

  → Threshold pair selected by the Sharpe criterion.

- `sharpe_calibration`

  → Sharpe ratio of the selected pair during calibration.

- `sharpe_calibration_return`

  → Calibration-period return of the selected pair.

#### Calmar

Selects the Stage-2 threshold combination with the highest **Calmar ratio**.

- `calmar_buy_thr`
- `calmar_sell_thr`

  → Threshold pair selected by the Calmar criterion.

- `calmar_calibration`

  → Calmar ratio of the selected pair during calibration.

- `calmar_calibration_return`

  → Calibration-period return of the selected pair.

#### Return / Risk

Selects the Stage-2 threshold combination with the highest **Return/Risk metric**.

- `return_risk_buy_thr`
- `return_risk_sell_thr`

  → Threshold pair selected by the Return/Risk criterion.

- `return_risk_calibration`

  → Return/Risk metric of the selected pair during calibration.

- `return_risk_calibration_return`

  → Calibration-period return of the selected pair.

#### Robust

Robust is different: it does **not simply select the single Stage-2 simulation with the highest value of one metric**.

Instead, it evaluates the Stage-2 results as a **top-performing region** and derives a threshold pair intended to be more stable than a single point estimate.

- `robust_buy_thr`
- `robust_sell_thr`

  → Final robust threshold pair derived from the Stage-2 top-performing region.

- `robust_calibration_sharpe`
- `robust_calibration_return`
- `robust_calibration_calmar`
- `robust_calibration_return_risk`

  → Performance metrics of the final robust threshold pair during calibration.

**Important:** Robust is a **threshold-stability selection method**, whereas Sharpe, Calmar and Return/Risk select the best individual Stage-2 candidate according to their respective objective.

---

### 5. OOS Results

After Stage 2 has selected the four threshold pairs, **each pair is applied separately to the next unseen test period**.

- `sharpe_oos_return`
- `calmar_oos_return`
- `return_risk_oos_return`
- `robust_oos_return`

→ Actual **OOS return** produced by each Stage-2-selected threshold pair.

These values are **not MC optimization scores**. They are the subsequent performance results obtained when the selected thresholds are tested on data that was not used during optimization.

This is the most important part for judging whether the optimization **generalizes out of sample**.

---

### 6. Excess Return vs INDEX

`index_return`

→ Return of the DOW/INDEX over the same OOS period.

For each method:

- `sharpe_excess_return`
- `calmar_excess_return`
- `return_risk_excess_return`
- `robust_excess_return`

are calculated as:

`strategy OOS return − index return`

Positive = the strategy outperformed the index during that OOS period.

---

### 7. Stage-2 Distribution

- `stage2_buy_std`
- `stage2_sell_std`

  → Standard deviation of the **Stage-2 sampled buy/sell thresholds**.

- `stage2_actual_buy_min`
- `stage2_actual_buy_max`
- `stage2_actual_sell_min`
- `stage2_actual_sell_max`

  → Actual minimum and maximum buy/sell thresholds generated among the Stage-2 simulations.

**Important:** these describe the **overall Stage-2 search/sample distribution**. They do not separately describe the Sharpe, Calmar, Return/Risk or Robust selected regions.

---

### In short

The complete process is:

**Stage 1 → identify promising threshold region → Stage 2 → evaluate the focused candidates using 4 selection criteria → select 4 threshold pairs → test all 4 pairs on unseen OOS data**

Conceptually:

**Stage 1**  
→ broad search

**Stage 2**  
→ focused search around Stage-1 promising region

**Four selections**  
→ Sharpe  
→ Calmar  
→ Return/Risk  
→ Robust

**OOS**  
→ independently test each selected threshold pair

The key comparison is therefore:

**`sharpe_oos_return` vs `calmar_oos_return` vs `return_risk_oos_return` vs `robust_oos_return` vs `index_return`**

And for relative performance:

**`sharpe_excess_return` vs `calmar_excess_return` vs `return_risk_excess_return` vs `robust_excess_return`**

The OOS results are the primary evidence for deciding whether the four selection approaches actually generalize beyond the calibration data.


In [ ]:
Historical data
      │
      ▼
Walk-forward calibration window
      │
      ▼
Stage 1 Monte-Carlo
      │
      │ Broad search
      ▼
Top-performing threshold region
      │
      ▼
Robust center + threshold dispersion
      │
      ▼
Stage 2 Monte-Carlo
      │
      │ Focused search
      ▼
 ┌──────────────┬──────────────┬──────────────┬──────────────┐
 │    Sharpe    │    Calmar    │ Return/Risk  │    Robust    │
 └──────────────┴──────────────┴──────────────┴──────────────┘
      │
      ▼
Four threshold pairs
      │
      ▼
Following OOS period
      │
      ├── Sharpe OOS
      ├── Calmar OOS
      ├── Return/Risk OOS
      └── Robust OOS
      │
      ▼
Compare against INDEX
      │
      ▼
Move calibration window forward
      │
      ▼
Repeat

In [ ]:
# Walk-Forward Optimization — Column Production Flow

```text
                    ┌──────────────────────────────┐
                    │ 1. WALK-FORWARD WINDOW       │
                    │                              │
                    │ calibration_start            │
                    │ calibration_end              │
                    │ test_start                   │
                    │ test_end                     │
                    └──────────────┬───────────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │ 2. STAGE 1 — BROAD MC        │
                    │                              │
                    │ Broad threshold search        │
                    │                              │
                    │ stage1_robust_buy_thr         │
                    │ stage1_robust_sell_thr        │
                    │ stage1_top10_buy_std          │
                    │ stage1_top10_sell_std         │
                    └──────────────┬───────────────┘
                                   │
                         promising region
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │ 3. STAGE 2 — FOCUSED MC      │
                    │                              │
                    │ Search around Stage-1 region  │
                    │                              │
                    │ stage2_buy_min / max          │
                    │ stage2_sell_min / max         │
                    │ stage2_n                      │
                    │ stage2_buy_std / sell_std     │
                    │ stage2_actual_*_min / max     │
                    └──────────────┬───────────────┘
                                   │
                                   ▼
             ┌─────────────────────────────────────────┐
             │ 4. SCORE STAGE-2 CANDIDATES             │
             │    on CALIBRATION data                  │
             └──────┬────────┬────────┬────────┬───────┘
                    │        │        │        │
                    ▼        ▼        ▼        ▼
              ┌─────────┐ ┌────────┐ ┌──────────┐ ┌─────────┐
              │ SHARPE  │ │ CALMAR │ │RETURN/   │ │ ROBUST  │
              │         │ │        │ │RISK      │ │         │
              │ *_buy   │ │ *_buy  │ │ *_buy    │ │ *_buy   │
              │ *_sell  │ │ *_sell │ │ *_sell   │ │ *_sell  │
              │ *_calib │ │ *_calib│ │ *_calib  │ │ *_calib │
              └────┬────┘ └───┬────┘ └────┬─────┘ └────┬────┘
                   │           │           │             │
                   └───────────┴───────────┴─────────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │ 5. OOS TEST                  │
                    │                              │
                    │ Apply the 4 selected pairs   │
                    │ to UNSEEN test data          │
                    │                              │
                    │ No further optimization      │
                    └──────────────┬───────────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │ 6. OOS RETURNS               │
                    │                              │
                    │ sharpe_oos_return            │
                    │ calmar_oos_return             │
                    │ return_risk_oos_return        │
                    │ robust_oos_return             │
                    └──────────────┬───────────────┘
                                   │
                    ┌──────────────┴──────────────┐
                    │                             │
                    ▼                             ▼
          ┌──────────────────┐          ┌──────────────────┐
          │ index_return     │          │ Strategy OOS     │
          │ (benchmark)      │          │ returns          │
          └────────┬─────────┘          └────────┬─────────┘
                   └──────────────┬──────────────┘
                                  ▼
                    ┌──────────────────────────────┐
                    │ 7. EXCESS RETURNS             │
                    │                              │
                    │ strategy OOS − index return │
                    │                              │
                    │ sharpe_excess_return         │
                    │ calmar_excess_return          │
                    │ return_risk_excess_return     │
                    │ robust_excess_return          │
                    └──────────────────────────────┘
```

## Column groups by process

| Process | Columns produced |
|---|---|
| **1. Window definition** | `calibration_start`, `calibration_end`, `test_start`, `test_end` |
| **2. Stage 1 MC** | `stage1_robust_buy_thr`, `stage1_robust_sell_thr`, `stage1_top10_buy_std`, `stage1_top10_sell_std` |
| **3. Stage 2 MC** | `stage2_buy_min`, `stage2_buy_max`, `stage2_sell_min`, `stage2_sell_max`, `stage2_n`, `stage2_buy_std`, `stage2_sell_std`, `stage2_actual_*` |
| **4a. Sharpe selection** | `sharpe_buy_thr`, `sharpe_sell_thr`, `sharpe_calibration`, `sharpe_calibration_return` |
| **4b. Calmar selection** | `calmar_buy_thr`, `calmar_sell_thr`, `calmar_calibration`, `calmar_calibration_return` |
| **4c. Return/Risk selection** | `return_risk_buy_thr`, `return_risk_sell_thr`, `return_risk_calibration`, `return_risk_calibration_return` |
| **4d. Robust selection** | `robust_buy_thr`, `robust_sell_thr`, `robust_calibration_sharpe`, `robust_calibration_return`, `robust_calibration_calmar`, `robust_calibration_return_risk` |
| **5–6. OOS test** | `sharpe_oos_return`, `calmar_oos_return`, `return_risk_oos_return`, `robust_oos_return` |
| **Benchmark** | `index_return` |
| **7. OOS comparison** | `sharpe_excess_return`, `calmar_excess_return`, `return_risk_excess_return`, `robust_excess_return` |
| **Diagnostics** | `max_num_components`, `mu_sigma` |

## The key distinction

**Stage 1 → Stage 2** determines *where to search*.

**Sharpe / Calmar / Return-Risk / Robust** determine *which threshold pair to take from that Stage-2 search*.

**OOS** determines *whether those choices actually generalize to unseen data*.

> **Most important research outputs:** the four `*_oos_return` columns and especially the four `*_excess_return` columns across all walk-forward windows.
